# Analysis of the book reading service database

<b>Project goal:</b> To analyze the database of a large book subscription service.

It contains information about books, publishers, authors, and user reviews. This data will help formulate a value proposition for the new product.

Let's download the necessary libraries and connect to the database:

In [1]:
import pandas as pd
from sqlalchemy import create_engine
# set the parameters
db_config = {'user': 'praktikum_student', # username
'pwd': 'Sdf4$2;d-d30pp', # password
'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
'port': 6432, # connection port
'db': 'data-analyst-final-project-db'} # database name
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
 db_config['pwd'],
 db_config['host'],
 db_config['port'],
 db_config['db'])
# save the connector
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

Let's take a look at all the tables in the database:

1. Table __books__:

In [2]:
query = """
SELECT *
FROM books
LIMIT 5;
"""
table = pd.io.sql.read_sql(query, con = engine)
table

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Contains book data:

- `book_id` — book identifier;
- `author_id` — author identifier;
- `title` — book title;
- `num_pages` — number of pages;
- `publication_date` — book publication date;
- `publisher_id` — publisher identifier.

2. Table __authors__:

In [3]:
query = """
SELECT *
FROM authors
LIMIT 5;
"""
table = pd.io.sql.read_sql(query, con = engine)
table

,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Contains author data:

- `author_id` — author ID;
- `author` — author name.

3. Table __ratings__:

In [4]:
query = """
SELECT *
FROM ratings
LIMIT 5;
"""
table = pd.io.sql.read_sql(query, con = engine)
table

,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Contains data about user book ratings:

- `rating_id` — rating ID;
- `book_id` — book ID;
- `username` — username of the user who left the rating;
- `rating` — book rating.

4. Table __reviews__:

In [5]:
query = """
SELECT *
FROM reviews
LIMIT 5;
"""
table = pd.io.sql.read_sql(query, con = engine)
table

,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


Contains data about user book reviews:

- `review_id` — review ID;
- `book_id` — book ID;
- `username` — username of the user who wrote the review;
- `text` — review text.

5. Table __publishers__:

In [6]:
query = """
SELECT *
FROM publishers
LIMIT 5;
"""
table = pd.io.sql.read_sql(query, con = engine)
table

,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Contains data about publishers:

- `publisher_id` — publisher ID;
- `publisher` — publisher name;

In [2]:
# what other tables are available in the database?
display(pd.io.sql.read_sql('''

SELECT *
FROM pg_catalog.pg_tables
WHERE schemaname != 'pg_catalog' AND
      schemaname != 'information_schema';

''', con = engine))


# look at the column types in the tables of interest.
display(pd.io.sql.read_sql('''
SELECT 
    table_name, 
    column_name, 
    data_type, 
    is_nullable
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE table_name IN ('books', 'authors', 'publishers', 'ratings', 'reviews');
''', con = engine))

,schemaname,tablename,tableowner,tablespace,hasindexes,hasrules,hastriggers,rowsecurity
0,public,author,praktikum_student,None,True,False,False,False
1,public,advertisment_costs,praktikum_admin,None,True,False,False,False
2,public,authors,praktikum_admin,None,True,False,True,False
3,public,books,praktikum_admin,None,True,False,True,False
4,public,reviews,praktikum_admin,None,True,False,True,False
5,public,ratings,praktikum_admin,None,True,False,True,False
6,public,visits,praktikum_admin,None,True,False,False,False
7,public,second,praktikum_student,None,False,False,False,False
8,public,second_b,praktikum_student,None,False,False,False,False
9,public,orders,praktikum_admin,None,True,False,False,False


,table_name,column_name,data_type,is_nullable
0,books,book_id,integer,NO
1,ratings,rating,integer,YES
2,reviews,review_id,integer,NO
3,reviews,book_id,integer,YES
4,books,num_pages,integer,YES
5,authors,author_id,integer,NO
6,books,publication_date,date,YES
7,books,publisher_id,integer,YES
8,publishers,publisher_id,integer,NO
9,books,author_id,integer,YES


### Let's count how many books were published after January 1, 2000.

In [8]:
query = """
    SELECT COUNT(book_id)
    FROM books
    WHERE publication_date > '2000-01-01'::date

"""
data = pd.io.sql.read_sql(query, con = engine)
data

,count
0,819


Since January 1, 2000, 819 books have been published.

#publication_date >= '2000-01-01'

### For each book, we will calculate the number of reviews and the average rating.

In [9]:
query = '''
    SELECT
        b.title,
        (
            SELECT COUNT(review_id) AS review_count
            FROM reviews
            WHERE book_id = b.book_id
        ),
        (
            SELECT ROUND(AVG(rating), 2)
            FROM ratings
            WHERE book_id = b.book_id
        ) AS rating_avg
    FROM books as b, reviews as r
    GROUP BY b.book_id
    ORDER by review_count DESC;
'''
data = pd.io.sql.read_sql(query, con = engine)
data

,title,review_count,rating_avg
0,Twilight (Twilight #1),7,3.66
1,Water for Elephants,6,3.98
2,The Glass Castle,6,4.21
3,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
4,The Curious Incident of the Dog in the Night-Time,6,4.08
...,...,...,...
995,Anne Rice's The Vampire Lestat: A Graphic Novel,0,3.67
996,The Natural Way to Draw,0,3.00
997,The Cat in the Hat and Other Dr. Seuss Favorites,0,5.00
998,Essential Tales and Poems,0,4.00


We see that film adaptations lead the way in terms of number of reviews. Conversely, the ratings for these books are quite low.

 ### Let's identify the publishing house that has released the largest number of books longer than 50 pages - this way we will exclude brochures from the analysis

In [10]:
query = """
    WITH publisher_books_count AS (
        SELECT publisher,
               COUNT(book_id) AS books_count
        FROM publishers AS p
        INNER JOIN books AS b ON p.publisher_id = b.publisher_id
        WHERE num_pages > 50
        GROUP BY publisher
        ORDER BY books_count DESC
        )

    SELECT publisher
    FROM publisher_books_count
    WHERE books_count = (SELECT MAX(books_count)
                       FROM publisher_books_count)

"""
data = pd.io.sql.read_sql(query, con = engine)
data

,publisher
0,Penguin Books


It turned out that the largest number of books thicker than 50 pages were published by Penguin Books.

### We will determine the author with the highest average book rating - we will only consider books with 50 or more ratings

In [11]:
query = """

    SELECT author,
           AVG(rating) AS rating_avg
    FROM authors AS a
    JOIN books AS b ON a.author_id = b.author_id
    JOIN ratings AS r ON b.book_id = r.book_id

    WHERE b.book_id IN (
                      SELECT book_id
                      FROM ratings 
                      GROUP BY book_id
                      HAVING COUNT(rating_id) >= 50
                      ORDER BY COUNT(rating_id) DESC)
    GROUP BY author
    ORDER BY rating_avg DESC
    LIMIT 1

"""
data = pd.io.sql.read_sql(query, con = engine)
data

,author,rating_avg
0,J.K. Rowling/Mary GrandPré,4.287097


Of course, it's J.K. Rowling, with illustrations by Mary Grandpré. The most popular book with the highest rating is Harry Potter. Consider also that there are several Harry Potter books, all top-rated, which increases the author's ratings.

### Let's calculate the average number of reviews from users who have given more than 50 ratings.

In [12]:
query = """
    SELECT ROUND(AVG(review_count), 1)
    FROM (
           SELECT username, 
                  COUNT(review_id) AS review_count
           FROM reviews 
           WHERE username IN (
                              SELECT username
                              FROM ratings
                              GROUP BY username
                              HAVING COUNT(rating_id) > 50)
           GROUP BY username
          ) AS review_count
"""
data = pd.io.sql.read_sql(query, con = engine)
data

,round
0,24.3


We see that the most reading users are also the most writing. On average, a user who has rated more than 50 books (and therefore read more than 50 books) has written more than 24 reviews. Where do they find so much time? They probably don't have Yandex.Practicum.